
# COVID-19 México — Análisis Topológico con Mapper

## Pipeline y de qué notebook sale CADA cosa

```
NOTEBOOK │ QUÉ USO                               │ PARA QUÉ
─────────┼───────────────────────────────────────┼──────────────────────────────────
N1       │ gudhi.RipsComplex, filtración          │ Ver la topología RAW de la nube
N2       │ ripser(), plot_diagrams()              │ Diagramas de persistencia → β₀, β₁
N3       │ BettiCurve (giotto-tda)                │ Cómo cambia la topología con la escala
N4       │ bottleneck_distance, wasserstein       │ Comparar topología entre periodos
N5       │ NO SE USA                              │ Es para series temporales (Takens), 
         │                                        │ nuestros datos son espaciales
N6       │ umap.UMAP, TSNE                        │ 1) Visualizar datos antes de Mapper
         │                                        │ 2) UMAP 1D como lens para fMapper
N7       │ build_mapper, graph_stats,             │ Pipeline principal de Mapper
         │ draw_mapper_graph, node_purity          │
```

### ¿Por qué NO se usa N5?
N5 es para **Takens Embedding** (encaje de retardo temporal): toma una serie 1D
$x_t$ y la reconstruye en ℝ^d. Los datos de COVID son **espaciales** (lat, lon,
casos, fecha), no una sola serie temporal escalar. Si quisiéramos analizar la
**serie temporal** de casos diarios como señal periódica, ahí sí usaríamos N5.
Pero el ejercicio pide Mapper sobre los datos espaciales/temporales.


---
## PASO 0 — Instalación e imports


In [ ]:

# PASO 0: INSTALACIÓN

# De N7 celda 1: kmapper, networkx
# De N1 celda 1: gudhi (para Rips complex y filtración)
# De N2 celda 1: ripser, persim (diagramas de persistencia)
# De N3 celda 1: giotto-tda (Betti Curves)
# De N6 celda 2: umap-learn (UMAP para visualización + lens)

%pip install -q "kmapper>=2.1.0" networkx gudhi ripser persim "giotto-tda" "umap-learn>=0.5.7"


In [ ]:

# IMPORTS — cada uno dice de qué notebook viene


import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import rcParams

# --- De N7 celda 2: Mapper ---
import kmapper as km
import networkx as nx
from sklearn.cluster import DBSCAN, MiniBatchKMeans
from sklearn.preprocessing import StandardScaler

# --- De N1 celda 1: Filtración ---
import gudhi as gd

# --- De N2 celda 1: Persistent Homology ---
from ripser import ripser
from persim import plot_diagrams

# --- De N3 celda 1: Betti Curves ---
from gtda.diagrams import BettiCurve
from gtda.homology import VietorisRipsPersistence

# --- De N4 celda 2: Métricas de Persistencia ---
from gudhi.wasserstein import wasserstein_distance

# --- De N6 celda 2: UMAP y t-SNE ---
import umap
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA

np.random.seed(42)
print("✅ Imports de N1, N2, N3, N4, N6, N7 cargados")


---
## Funciones del curso — de qué celda exacta salen


In [ ]:

# FUNCIONES DE N7 CELDA 5 — 4 funciones base de Mapper


# N7 celda 5, función 1
def mapper_to_networkx(graph_dict):
    G = nx.Graph()
    for node, members in graph_dict["nodes"].items():
        G.add_node(node, size=len(members))
    for src, dsts in graph_dict["links"].items():
        for dst in dsts:
            G.add_edge(src, dst)
    return G

# N7 celda 5, función 2 — β₁ = M - N + C (Euler)
def graph_stats(G):
    n = G.number_of_nodes()
    m = G.number_of_edges()
    c = nx.number_connected_components(G) if n > 0 else 0
    cycles = m - n + c
    avg_deg = float(np.mean([d for _, d in G.degree()])) if n > 0 else 0.0
    return {"nodes": n, "edges": m, "components": c,
            "cycles": int(cycles), "avg_degree": avg_deg}

# N7 celda 5, función 3
def draw_mapper_graph(G, title, node_color="#3f72af"):
    fig, ax = plt.subplots(figsize=(10, 7))
    if G.number_of_nodes() == 0:
        ax.text(0.5, 0.5, "Grafo vacío — ajusta eps", ha="center", va="center")
        ax.set_title(title, fontweight='bold'); plt.show(); return
    pos = nx.spring_layout(G, seed=7)
    sizes = [55 + 2.5 * G.nodes[n]["size"] for n in G.nodes()]
    nx.draw_networkx_edges(G, pos, ax=ax, alpha=0.35, width=1.0, edge_color="#555555")
    nx.draw_networkx_nodes(G, pos, ax=ax, node_size=sizes, node_color=node_color, alpha=0.95)
    ax.set_title(title, fontweight='bold', fontsize=13)
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()

# N7 celda 5, función 4 — PIPELINE COMPLETO
def build_mapper(X, lens, n_cubes=10, overlap=0.3, clusterer=None):
    mapper = km.KeplerMapper(verbose=0)
    if clusterer is None:
        clusterer = DBSCAN(eps=0.25, min_samples=5)
    cover = km.Cover(n_cubes=n_cubes, perc_overlap=overlap)
    graph = mapper.map(lens, X, clusterer=clusterer, cover=cover)
    Gnx = mapper_to_networkx(graph)
    return graph, Gnx

# N7 celda 10 — node purity
def node_purity(graph_dict, labels):
    purities = []; details = []
    for node, idx in graph_dict["nodes"].items():
        node_labels = labels[idx]
        counts = np.bincount(node_labels, minlength=len(np.unique(labels)))
        purity = counts.max() / counts.sum()
        purities.append((len(idx), purity))
        details.append((node, len(idx), purity, counts.tolist()))
    if not purities:
        return 0.0, pd.DataFrame(columns=["node","size","purity","label_counts"])
    weighted = np.average([p for _,p in purities], weights=[s for s,_ in purities])
    detail_df = pd.DataFrame(details, columns=["node","size","purity","label_counts"])
    detail_df = detail_df.sort_values(["purity","size"], ascending=[True,False])
    return weighted, detail_df

# N4 celda 2 — helper para filtrar puntos finitos de un diagrama
def finite_points(dgm):
    if len(dgm) == 0: return np.zeros((0, 2))
    return dgm[np.isfinite(dgm[:, 1])]

print("✅ Funciones cargadas:")
print("   De N7: mapper_to_networkx, graph_stats, draw_mapper_graph, build_mapper, node_purity")
print("   De N4: finite_points")


---
## PASO 1 — Cargar y limpiar datos COVID


In [ ]:

# PASO 1.1: Leer CSV


df_raw = pd.read_csv("DB/COVID19MEXICO.csv")
print(f"Filas: {len(df_raw):,}")
df_raw.head(2)


In [ ]:

# PASO 1.2: Filtrar SOLO confirmados
# CLASIFICACION_FINAL ∈ {1, 2, 3} = confirmado


df_conf = df_raw[df_raw["CLASIFICACION_FINAL"].isin([1, 2, 3])].copy()
print(f"Confirmados: {len(df_conf):,} de {len(df_raw):,} ({len(df_conf)/len(df_raw)*100:.1f}%)")


In [ ]:

# PASO 1.3: Catálogo de estados (código INEGI → nombre + lat + lon)


ESTADOS = {
    1:("Aguascalientes",21.88,-102.29), 2:("Baja California",30.84,-115.28),
    3:("Baja California Sur",24.14,-110.31), 4:("Campeche",19.83,-90.53),
    5:("Coahuila",27.06,-101.71), 6:("Colima",19.24,-103.72),
    7:("Chiapas",16.75,-93.12), 8:("Chihuahua",28.63,-106.09),
    9:("Ciudad de México",19.43,-99.13), 10:("Durango",24.02,-104.66),
    11:("Guanajuato",21.02,-101.26), 12:("Guerrero",17.44,-99.55),
    13:("Hidalgo",20.09,-98.76), 14:("Jalisco",20.66,-103.35),
    15:("Estado de México",19.29,-99.66), 16:("Michoacán",19.70,-101.19),
    17:("Morelos",18.68,-99.10), 18:("Nayarit",21.75,-104.85),
    19:("Nuevo León",25.67,-100.32), 20:("Oaxaca",17.07,-96.73),
    21:("Puebla",19.04,-98.21), 22:("Querétaro",20.59,-100.39),
    23:("Quintana Roo",19.18,-88.48), 24:("San Luis Potosí",22.15,-100.98),
    25:("Sinaloa",24.77,-107.39), 26:("Sonora",29.07,-110.96),
    27:("Tabasco",17.99,-92.95), 28:("Tamaulipas",24.27,-98.84),
    29:("Tlaxcala",19.32,-98.24), 30:("Veracruz",19.17,-96.13),
    31:("Yucatán",20.97,-89.59), 32:("Zacatecas",22.77,-102.58),
}

df_conf["NOMBRE_ESTADO_RES"] = df_conf["ENTIDAD_RES"].map(lambda x: ESTADOS.get(x,("Desc",0,0))[0])
df_conf["LAT_ESTADO_RES"]    = df_conf["ENTIDAD_RES"].map(lambda x: ESTADOS.get(x,("",0,0))[1])
df_conf["LON_ESTADO_RES"]    = df_conf["ENTIDAD_RES"].map(lambda x: ESTADOS.get(x,("",0,0))[2])
df_conf["NOMBRE_ESTADO_NAC"] = df_conf["ENTIDAD_NAC"].map(lambda x: ESTADOS.get(x,("Desc",0,0))[0])

# Fecha → Día numérico
df_conf["FECHA_INGRESO"] = pd.to_datetime(df_conf["FECHA_INGRESO"], errors="coerce")
df_conf = df_conf.dropna(subset=["FECHA_INGRESO"])
fecha_min = df_conf["FECHA_INGRESO"].min()
df_conf["DIA_NUM"] = (df_conf["FECHA_INGRESO"] - fecha_min).dt.days

print(f"✅ Rango: {fecha_min.date()} → {df_conf['FECHA_INGRESO'].max().date()}")


In [ ]:

# PASO 1.4: Agregar por (Día, Estado, Municipio) → NUM_CASOS


df_agg = (
    df_conf.groupby(["DIA_NUM","ENTIDAD_RES","MUNICIPIO_RES",
                      "NOMBRE_ESTADO_RES","LAT_ESTADO_RES","LON_ESTADO_RES"])
    .size().reset_index(name="NUM_CASOS")
)
df_agg["NOMBRE_MUNICIPIO"] = df_agg["NOMBRE_ESTADO_RES"]+"-Mun"+df_agg["MUNICIPIO_RES"].astype(str).str.zfill(3)
df_agg["LAT_MUNICIPIO"] = df_agg["LAT_ESTADO_RES"]
df_agg["LON_MUNICIPIO"] = df_agg["LON_ESTADO_RES"]

print(f"Tabla agregada: {len(df_agg):,} filas")
df_agg.head()


In [ ]:

# PASO 1.5: Preparar X y lens — normalizar (N7 celda 5 demo)


feature_cols = ["DIA_NUM","NUM_CASOS","LAT_ESTADO_RES","LON_ESTADO_RES",
                "LAT_MUNICIPIO","LON_MUNICIPIO"]
lens_cols = ["DIA_NUM","NUM_CASOS","LAT_ESTADO_RES","LON_ESTADO_RES"]

X_raw = df_agg[feature_cols].values.astype(float)
scaler = StandardScaler()       # ← N7 celda 5: "estandarizar, media=0, std=1"
X = scaler.fit_transform(X_raw)
lens = X[:, [0,1,2,3]]          # las 4 columnas de la proyección

print(f"X: {X.shape}, lens: {lens.shape}")
print("Normalizado ✅")


---
## PASO 2 — Visualizar los datos ANTES de Mapper (N6: UMAP + t-SNE + PCA)

De **N6 celda 12** (UMAP) y **N6 celda 8** (t-SNE):
> "PCA gives linear projections. But many datasets live on nonlinear manifolds."

Primero vemos la **forma** de los datos en 2D con tres métodos.
Esto nos da intuición sobre cuántos clusters hay ANTES de aplicar Mapper.


In [ ]:

# PASO 2.1: PCA, t-SNE y UMAP en 2D — de N6 celdas 8 y 12

# Submuestrear si hay muchas filas (para que t-SNE/UMAP no tarden horas)

MAX_POINTS = 5000
if len(X) > MAX_POINTS:
    idx_sub = np.random.choice(len(X), MAX_POINTS, replace=False)
    X_vis = X[idx_sub]
    estados_vis = df_agg["ENTIDAD_RES"].values[idx_sub]
else:
    X_vis = X
    estados_vis = df_agg["ENTIDAD_RES"].values
    idx_sub = np.arange(len(X))

# PCA 2D — de N6 celda 8: PCA(n_components=2).fit_transform(X)
X_pca = PCA(n_components=2).fit_transform(X_vis)

# t-SNE 2D — de N6 celda 8: TSNE(perplexity=30, learning_rate='auto')
X_tsne = TSNE(n_components=2, perplexity=30, learning_rate='auto',
              random_state=42).fit_transform(X_vis)

# UMAP 2D — de N6 celda 12: umap.UMAP(n_neighbors=15, min_dist=0.1)
X_umap = umap.UMAP(n_neighbors=15, min_dist=0.1,
                    random_state=42).fit_transform(X_vis)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (Xr, name) in zip(axes, [(X_pca, "PCA"), (X_tsne, "t-SNE"), (X_umap, "UMAP")]):
    sc = ax.scatter(Xr[:,0], Xr[:,1], c=estados_vis, cmap='tab20',
                    s=5, alpha=0.6)
    ax.set_title(f"{name} 2D — COVID (color=estado)", fontweight='bold')
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("INTERPRETACIÓN (de N6):")
print("  - PCA: proyección lineal → aplasta estructura no-lineal")
print("  - t-SNE: preserva vecindarios locales → clusters nítidos")
print("  - UMAP: balance local/global → estructura más fiel")
print("  Observa cuántos grupos naturales se ven ANTES de Mapper")


---
## PASO 3 — Homología Persistente de la nube de puntos (N2: Ripser)

De **N2 celda 2**: usamos `ripser()` para calcular diagramas de persistencia.
Esto nos dice la topología REAL de la nube: cuántas componentes (β₀) y
cuántos loops (β₁) hay en los datos, INDEPENDIENTE de Mapper.


In [ ]:

# PASO 3.1: Diagrama de persistencia con Ripser — de N2 celda 6-8

# Submuestrear para eficiencia (Ripser es O(n³) en memoria)

RIPSER_SAMPLE = min(1500, len(X))
idx_rip = np.random.choice(len(X), RIPSER_SAMPLE, replace=False)
X_rip = X[idx_rip]

# ripser() — N2 celda 8: ripser(data, maxdim=1)["dgms"]
result = ripser(X_rip, maxdim=1)
dgms = result["dgms"]

# Visualizar — N2 celda 8: plot_diagrams(dgms)
fig, ax = plt.subplots(figsize=(6, 6))
plot_diagrams(dgms, show=False, ax=ax)
ax.set_title("Diagrama de persistencia — COVID datos crudos\n"
             "(N2: Ripser)", fontweight='bold')
plt.tight_layout()
plt.show()

# Contar features persistentes — N2 celda 9
h0_long = len([p for p in dgms[0] if (p[1]-p[0]) > 0.5])
h1_pts = finite_points(dgms[1])   # ← N4 celda 2
h1_long = len([p for p in h1_pts if (p[1]-p[0]) > 0.3])

print(f"H₀ features largos (persistencia > 0.5): {h0_long}")
print(f"  → sugiere ≈ {h0_long} componentes conexas naturales")
print(f"H₁ features largos (persistencia > 0.3): {h1_long}")
print(f"  → sugiere ≈ {h1_long} loops/ciclos en los datos")


---
## PASO 4 — Betti Curves (N3: giotto-tda)

De **N3 celda 17-18**: la Betti Curve muestra $\beta_q(t)$ = cuántos features
están vivos en cada valor de filtración $t$.

Esto nos permite ver a QUÉ ESCALA aparecen y desaparecen los clusters y loops.


In [ ]:

# PASO 4.1: Betti Curves — de N3 celda 18

# VietorisRipsPersistence → formato giotto-tda → BettiCurve

# Necesitamos shape (n_samples, n_points, n_features)
X_gtda = X_rip.reshape(1, *X_rip.shape)  # 1 "muestra" = toda la nube

# N3 celda 12/N2: VietorisRipsPersistence
vr = VietorisRipsPersistence(metric="euclidean", homology_dimensions=[0, 1], n_jobs=1)
gtda_dgms = vr.fit_transform(X_gtda)

# N3 celda 18: BettiCurve(n_bins=...)
betti_fine = BettiCurve(n_bins=100)
B = betti_fine.fit_transform(gtda_dgms)

print(f"Betti curve shape: {B.shape}")
# B tiene shape (1, n_homology_dims * n_bins)

# Separar H0 y H1
n_bins = 100
B_flat = B[0]  # (200,) = 100 bins H0 + 100 bins H1
B_h0 = B_flat[:n_bins]
B_h1 = B_flat[n_bins:2*n_bins]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(B_h0, color="#3f72af", lw=2)
axes[0].set_title("Betti Curve β₀ — componentes conexas", fontweight='bold')
axes[0].set_xlabel("Valor de filtración (bins)")
axes[0].set_ylabel("β₀(t)")
axes[0].grid(True, alpha=0.3)

axes[1].plot(B_h1, color="#e76f51", lw=2)
axes[1].set_title("Betti Curve β₁ — loops", fontweight='bold')
axes[1].set_xlabel("Valor de filtración (bins)")
axes[1].set_ylabel("β₁(t)")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("INTERPRETACIÓN (de N3 celda 24):")
print("  β₀: empieza alto (muchos puntos aislados) → baja (se conectan)")
print("  β₁: pico indica loops formándose → baja cuando los loops se llenan")


---
## PASO 5 — Mapper (N7: pipeline principal)

Parámetros del ejercicio:
- `n_cubes=10`, `overlap=0.08` (8%), `DBSCAN`, métrica Euclidiana


In [ ]:

# PASO 5.1: Mapper — de N7 celda 5 (build_mapper)


graph_covid, G_covid = build_mapper(
    X, lens,
    n_cubes=10,
    overlap=0.08,
    clusterer=DBSCAN(eps=0.5, min_samples=3)
)

stats = graph_stats(G_covid)
print("ESTADÍSTICAS MAPPER COVID-19:")
for k,v in stats.items(): print(f"  {k}: {v}")
print(f"  Euler: β₁ = {stats['edges']} - {stats['nodes']} + {stats['components']} = {stats['cycles']}")


In [ ]:

# PASO 5.2: Visualizar grafo — de N7 celda 5 (draw_mapper_graph)


draw_mapper_graph(G_covid,
    f"Mapper COVID-19\nn_cubes=10, overlap=8%, DBSCAN\n"
    f"β₀={stats['components']}, β₁={stats['cycles']}, "
    f"N={stats['nodes']}, M={stats['edges']}",
    node_color="#e76f51")


In [ ]:

# PASO 5.3: Barrido de eps — de N7 celda 6 (hyperparameter study)

# "Higher n_cubes often increases graph detail (and noise)"
# "Higher overlap can connect nearby local clusters"

print("Barrido de eps (n_cubes=10, overlap=8%):")
print(f"{'eps':<6} {'nodes':>6} {'edges':>6} {'β₀':>5} {'β₁':>5}")
print("-"*35)

best_eps, best_nodes = 0.5, 0
for eps_val in [0.3, 0.5, 0.75, 1.0, 1.5, 2.0, 3.0]:
    _, Gt = build_mapper(X, lens, n_cubes=10, overlap=0.08,
                         clusterer=DBSCAN(eps=eps_val, min_samples=3))
    s = graph_stats(Gt)
    print(f"{eps_val:<6.1f} {s['nodes']:>6} {s['edges']:>6} {s['components']:>5} {s['cycles']:>5}")
    if s['nodes'] > best_nodes and s['nodes'] < 500:
        best_nodes = s['nodes']; best_eps = eps_val

print(f"\n→ Mejor eps para grafo informativo: {best_eps}")


In [ ]:

# PASO 5.4: Reconstruir con el mejor eps y visualizar


graph_best, G_best = build_mapper(
    X, lens, n_cubes=10, overlap=0.08,
    clusterer=DBSCAN(eps=best_eps, min_samples=3)
)
stats_best = graph_stats(G_best)

draw_mapper_graph(G_best,
    f"Mapper COVID-19 (eps={best_eps})\n"
    f"β₀={stats_best['components']}, β₁={stats_best['cycles']}, "
    f"N={stats_best['nodes']}, M={stats_best['edges']}",
    node_color="#e76f51")


In [ ]:

# PASO 5.5: Node purity por estado — de N7 celda 10


estado_labels = df_agg["ENTIDAD_RES"].values
unique_labs = np.unique(estado_labels)
lab_map = {v:i for i,v in enumerate(unique_labs)}
labels_int = np.array([lab_map[l] for l in estado_labels])

pur, detail = node_purity(graph_best, labels_int)
print(f"Pureza ponderada por estado: {pur:.3f}")
print(f"  1.0 = cada nodo tiene UN solo estado")
print(f"  <0.5 = nodos muy mixtos geográficamente\n")
print("Top 10 nodos más mixtos:")
detail.head(10)


---
## PASO 6 — fMapper con UMAP como lens (N7 sección 4 + N6)

De **N7 celda 13** (fMapper): 
> "fMapper means a function-guided Mapper variant where the lens f is
> learned from data (instead of using a raw coordinate)."

De **N6 celda 12**: UMAP puede reducir a 1D o 2D de forma no-lineal.
Combinando ambos → fMapper con lens aprendida.

Esto responde directamente la **Pregunta 4**: "¿Qué tipo de proyección
daría información más relevante?"


In [ ]:

# PASO 6.1: fMapper — de N7 celda 13 + N6 celda 12

# Lens aprendida con UMAP (1D) en vez de coordenadas crudas

# N6 celda 12: umap.UMAP(n_neighbors=15, min_dist=0.1)
lens_umap = umap.UMAP(n_components=1, n_neighbors=15, min_dist=0.1,
                       random_state=42).fit_transform(X)

# N7 celda 13: build_mapper con la lens aprendida
# N7 usa MiniBatchKMeans para fMapper (celda 13)
graph_fmapper, G_fmapper = build_mapper(
    X, lens_umap,
    n_cubes=10, overlap=0.08,
    clusterer=DBSCAN(eps=best_eps, min_samples=3)
)

stats_fm = graph_stats(G_fmapper)

# Comparar lado a lado — como N7 celda 14
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, (G, title, color) in zip(axes, [
    (G_best, f"Mapper (coord lens)\nβ₀={stats_best['components']}, β₁={stats_best['cycles']}", "#3f72af"),
    (G_fmapper, f"fMapper (UMAP lens)\nβ₀={stats_fm['components']}, β₁={stats_fm['cycles']}", "#2a9d8f")
]):
    if G.number_of_nodes() > 0:
        pos = nx.spring_layout(G, seed=7)
        sizes = [55 + 2.5 * G.nodes[n]["size"] for n in G.nodes()]
        nx.draw_networkx_edges(G, pos, ax=ax, alpha=0.35, width=1.0)
        nx.draw_networkx_nodes(G, pos, ax=ax, node_size=sizes, node_color=color, alpha=0.95)
    ax.set_title(title, fontweight='bold', fontsize=12)
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(True, alpha=0.3)

plt.suptitle("Mapper vs fMapper — COVID-19", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


---
## PASO 7 — Comparar topología entre periodos (N4: Wasserstein)

De **N4 celda 10**: la distancia Wasserstein mide qué tan diferentes
son dos diagramas de persistencia.

Dividimos los datos en dos periodos y comparamos si la topología cambió
entre la primera y segunda mitad de la pandemia.


In [ ]:

# PASO 7.1: Wasserstein entre dos periodos — de N4 celda 10-11


dia_mediana = int(np.median(df_agg["DIA_NUM"]))
mask_1 = df_agg["DIA_NUM"].values <= dia_mediana
mask_2 = df_agg["DIA_NUM"].values > dia_mediana

# Submuestrear cada mitad
n_sub = min(800, mask_1.sum(), mask_2.sum())
idx1 = np.random.choice(np.where(mask_1)[0], n_sub, replace=False)
idx2 = np.random.choice(np.where(mask_2)[0], n_sub, replace=False)

# Ripser en cada periodo — N2
dgm_p1 = ripser(X[idx1], maxdim=1)["dgms"]
dgm_p2 = ripser(X[idx2], maxdim=1)["dgms"]

# Wasserstein — N4 celda 10
h1_p1 = finite_points(dgm_p1[1])
h1_p2 = finite_points(dgm_p2[1])

w_dist = wasserstein_distance(h1_p1, h1_p2, order=1, internal_p=2)

# Bottleneck — N4 celda 8
b_dist = gd.bottleneck_distance(h1_p1, h1_p2, e=0.0)

print(f"Comparación topológica entre periodos:")
print(f"  Periodo 1: días 0-{dia_mediana} ({n_sub} muestras)")
print(f"  Periodo 2: días {dia_mediana+1}+ ({n_sub} muestras)")
print(f"\n  Wasserstein-1 (H₁): {w_dist:.4f}")
print(f"  Bottleneck   (H₁): {b_dist:.4f}")
print(f"\n  {'TOPOLOGÍA SIMILAR' if w_dist < 0.5 else 'TOPOLOGÍA DIFERENTE'}")
print(f"  → {'La pandemia mantuvo estructura similar' if w_dist < 0.5 else 'La pandemia CAMBIÓ de estructura'}")

# Visualizar ambos diagramas
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
plot_diagrams(dgm_p1, show=False, ax=axes[0])
axes[0].set_title(f"Periodo 1 (días 0-{dia_mediana})", fontweight='bold')
plot_diagrams(dgm_p2, show=False, ax=axes[1])
axes[1].set_title(f"Periodo 2 (días {dia_mediana+1}+)", fontweight='bold')
plt.suptitle(f"Wasserstein={w_dist:.3f}, Bottleneck={b_dist:.3f}", fontsize=12)
plt.tight_layout()
plt.show()


---
## PASO 8 — Gráficas para el reporte + respuestas a las 5 preguntas


In [ ]:

# Curva temporal + top estados (para Pregunta 1)


fig, axes = plt.subplots(1, 2, figsize=(14, 5))

casos_dia = df_conf.groupby("FECHA_INGRESO").size()
axes[0].plot(casos_dia.index, casos_dia.values, lw=0.8, color="#3f72af")
axes[0].set_title("Casos confirmados por día", fontweight='bold')
axes[0].set_xlabel("Fecha"); axes[0].set_ylabel("Casos"); axes[0].grid(True, alpha=0.3)

top_est = df_conf.groupby("NOMBRE_ESTADO_RES").size().sort_values(ascending=True).tail(10)
top_est.plot.barh(ax=axes[1], color="#e76f51")
axes[1].set_title("Top 10 estados", fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()


---
## Respuestas a las 5 preguntas

### 1. ¿Qué puedes decir de la pandemia en México?
- Curva temporal muestra olas de contagio (picos y valles)
- Estados más afectados: CDMX, Edo. Méx., Jalisco (zonas densamente pobladas)
- Estacionalidad visible en β₁ de la Betti Curve

### 2. ¿Qué puedes decir de los clusters formados por Mapper?
- β₀ = componentes desconectadas → "regímenes" distintos (N7 celda 19)
- β₁ = ciclos → posible periodicidad en la propagación
- Nodos grandes = concentraciones de muchos casos (zonas de alta incidencia)
- Componentes separadas pueden reflejar fases distintas de la pandemia

### 3. ¿Qué otras características para mejorar?
- Comorbilidades: DIABETES, HIPERTENSION, OBESIDAD (están en el CSV)
- Mortalidad: FECHA_DEF ≠ 9999-99-99
- TIPO_PACIENTE (ambulatorio vs hospitalizado)
- Densidad poblacional del municipio

### 4. ¿Qué tipo de proyección daría más información?
- **UMAP 1D como lens → fMapper** (N7 sección 4 + N6): captura geometría no-lineal
- Comparamos Mapper vs fMapper en el PASO 6 → ver cuál da mejor β₀/β₁
- PCA como lens es la alternativa lineal (N7 celda 9: "PCA baseline")

### 5. ¿Qué preguntas te gustaría responder?
- ¿Qué municipios fueron puentes de contagio entre estados? (nodos hub)
- ¿La topología de la pandemia cambió entre periodos? (lo respondimos con Wasserstein en PASO 7)
- ¿Qué comorbilidades definen clusters de alta mortalidad?
- ¿Hay periodicidad en las olas? (β₁ de Mapper temporal)


---
## Tabla resumen: qué usé de cada notebook

| Paso | Notebook | Función/Concepto | Celda exacta |
|------|----------|-----------------|--------------|
| Visualización previa | **N6** | `umap.UMAP`, `TSNE`, `PCA` | N6 celdas 8, 12 |
| Persistencia raw | **N2** | `ripser()`, `plot_diagrams()` | N2 celdas 6-8 |
| Betti Curves | **N3** | `BettiCurve`, `VietorisRipsPersistence` | N3 celdas 12, 18 |
| Comparar periodos | **N4** | `wasserstein_distance`, `bottleneck_distance` | N4 celdas 8, 10 |
| Normalizar | **N7** | `StandardScaler` | N7 celda 5 demo |
| Mapper pipeline | **N7** | `build_mapper`, `graph_stats`, `draw_mapper_graph` | N7 celda 5 |
| Barrido params | **N7** | sweep de eps | N7 celda 6 |
| Node purity | **N7** | `node_purity()` | N7 celda 10 |
| fMapper | **N7+N6** | `build_mapper` + `umap.UMAP` 1D lens | N7 celda 13 + N6 celda 12 |
| Helper | **N4** | `finite_points()` | N4 celda 2 |
| **NO SE USA** | **N1** | Filtración manual paso a paso | Innecesario: Ripser ya lo hace |
| **NO SE USA** | **N5** | Takens Embedding | Datos espaciales, no serie temporal |
